# 00 CNN Data Preparation

This notebook reuses the frozen MLP manifests when available, extracts one coefficient-by-time MFCC map per recording, fits per-coefficient normalisation on training data only, and writes memory-mapped CNN caches. It performs no model training and computes no test metrics.


## 1. Package Setup


In [1]:
# Purpose: Package installs are documented but not run automatically during this static refactor.
# %pip install tensorflow scikit-learn pandas numpy matplotlib seaborn joblib soundfile librosa


## 2. Imports, Configuration and Paths


In [2]:
import os
from pathlib import Path

# Purpose: Mounts Google Drive when this notebook is running in Google Colab.
# Why this exists: the project files, cached MFCC features, manifests, models, figures,
# and metric outputs live in Google Drive during Colab runs. The /content/drive path
# only represents the real MyDrive files after drive.mount("/content/drive") succeeds.
# Important: the project root should be the folder that contains Data, Model Variants,
# and outputs. For this project, that expected Colab folder is the INM701 folder below.
COLAB_DRIVE_MOUNT_POINT = Path("/content/drive")
EXPECTED_COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")

try:
    from google.colab import drive

    drive.mount(str(COLAB_DRIVE_MOUNT_POINT))
    print("Google Colab detected. Google Drive mounted.")

    current_project_root = os.environ.get("INTRO_AI_PROJECT_ROOT")
    current_data_dir_exists = bool(current_project_root) and (Path(current_project_root) / "Data").exists()
    expected_data_dir_exists = (EXPECTED_COLAB_PROJECT_ROOT / "Data").exists()

    # Purpose: Keep a valid user-provided project root, but repair stale runtime state
    # if a previous cell pointed INTRO_AI_PROJECT_ROOT somewhere that does not contain Data.
    if current_data_dir_exists:
        print("INTRO_AI_PROJECT_ROOT already points to a folder with Data, so it was kept.")
    elif expected_data_dir_exists:
        os.environ["INTRO_AI_PROJECT_ROOT"] = str(EXPECTED_COLAB_PROJECT_ROOT)
        print("INTRO_AI_PROJECT_ROOT set to the expected INM701 project folder.")
    elif not current_project_root:
        os.environ["INTRO_AI_PROJECT_ROOT"] = str(EXPECTED_COLAB_PROJECT_ROOT)
        print("INTRO_AI_PROJECT_ROOT was not set, so it now points to the expected INM701 folder.")
    else:
        print("INTRO_AI_PROJECT_ROOT was kept, but Data was not found there or in the expected INM701 folder.")
except Exception as exc:
    print("Google Colab Drive mount skipped. This is expected outside Colab.")
    print("Mount skip reason:", exc)

active_project_root = os.environ.get("INTRO_AI_PROJECT_ROOT", "not set")
print("INTRO_AI_PROJECT_ROOT:", active_project_root)
if active_project_root != "not set":
    active_project_root = Path(active_project_root)
    print("Project root exists:", active_project_root.exists())
    print("Expected Data folder:", active_project_root / "Data")
    print("Data folder exists:", (active_project_root / "Data").exists())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Colab detected. Google Drive mounted.
INTRO_AI_PROJECT_ROOT set to the expected INM701 project folder.
INTRO_AI_PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Education/INM701
Project root exists: True
Expected Data folder: /content/drive/MyDrive/Colab Notebooks/Education/INM701/Data
Data folder exists: True


In [3]:
# Purpose: 2. Imports, Configuration and Paths.
import json
import os
import random
import shutil
import warnings
from pathlib import Path

import joblib
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

SAMPLE_RATE = 22050
FIXED_DURATION_SECONDS = 5.0
N_MFCC = 40
N_FFT = 1024
HOP_LENGTH = int(round(SAMPLE_RATE * 0.010))
WIN_LENGTH = int(round(SAMPLE_RATE * 0.025))
TARGET_SAMPLES = int(round(SAMPLE_RATE * FIXED_DURATION_SECONDS))
AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}
FINAL_SPLIT_VERSION = "mlp_final_split_2026_08_24_v1"
RUN_FULL_CNN_DATA_PREPARATION = True
REUSE_MLP_SPLIT = True
MAX_FILES_PER_CLASS = None


def resolve_project_root():
    # Purpose: default path.
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    # Purpose: Runs each candidate configuration under the same data split for a fair validation comparison.
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


def first_existing_path(candidates):
    # Purpose: none exist.
    expanded = [Path(item).expanduser() for item in candidates if item not in [None, ""]]
    # Purpose: Iterates over this collection to build the next table, feature set, or experiment result
    # Purpose: consistently.
    for path in expanded:
        if path.exists():
            return path
    return expanded[0]


# Purpose: Centralizes filesystem paths so dataset inputs, caches, figures, models, and metric tables are easy
# Purpose: to trace.
PROJECT_ROOT = resolve_project_root()
DATA_ROOT = PROJECT_ROOT / "Data"
SYNTHETIC_AUDIO_DIR = first_existing_path([
    os.environ.get("MLAAD_SYNTHETIC_AUDIO_DIR", ""),
    DATA_ROOT / "MLAAD_10pct" / "fake",
    DATA_ROOT / "Unprocessed" / "MLAAD_10pct" / "fake",
    DATA_ROOT / "MLAAD_10pct",
    PROJECT_ROOT / "MLAAD_10pct" / "fake",
    PROJECT_ROOT / "MLAAD_10pct",
])
BONA_FIDE_AUDIO_DIR = first_existing_path([
    os.environ.get("MLAAD_BONA_FIDE_AUDIO_DIR", ""),
    DATA_ROOT / "genuine_audio",
    DATA_ROOT / "bona_fide",
    DATA_ROOT / "Unprocessed" / "genuine_audio",
    PROJECT_ROOT / "genuine_audio",
    PROJECT_ROOT / "bona_fide",
])
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cnn"
SOURCE_MLP_MANIFESTS_DIR = PROJECT_ROOT / "outputs" / "mlp" / "manifests"
CACHE_DIR = OUTPUT_DIR / "cache"
MANIFESTS_DIR = OUTPUT_DIR / "manifests"
CONFIGS_DIR = OUTPUT_DIR / "configs"
TABLES_DIR = OUTPUT_DIR / "tables"
HISTORIES_DIR = OUTPUT_DIR / "histories"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
PILOT_LEGACY_DIR = OUTPUT_DIR / "pilot_legacy"
# Purpose: Creates each output directory before later cells try to save tables, figures, or models.
for directory in [OUTPUT_DIR, CACHE_DIR, MANIFESTS_DIR, CONFIGS_DIR, TABLES_DIR, HISTORIES_DIR, METRICS_DIR, FIGURES_DIR, MODELS_DIR, PILOT_LEGACY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Run full preparation:", RUN_FULL_CNN_DATA_PREPARATION)


Project root: /content/drive/MyDrive/Colab Notebooks/Education/INM701
Run full preparation: True


## 3. Corrected Metadata Scanner


In [4]:
# Purpose: 3. Corrected Metadata Scanner.
MANIFEST_COLUMNS = ["path", "relative_path", "label", "class_name", "language", "tts_generator", "file_size_mb", "metadata_warning"]


def parse_synthetic_metadata(path, root_dir):
    # Purpose: Extracts synthetic-language and generator labels from the dataset folder structure.
    parts = path.relative_to(root_dir).parts
    warning = ""
    if parts and parts[0].lower() == "fake" and len(parts) >= 3:
        language, tts_generator = parts[1], parts[2]
    elif len(parts) >= 2:
        language, tts_generator = parts[0], parts[1]
    else:
        language, tts_generator = "unknown", "unknown"
        warning = "synthetic language/generator not derivable from path"
    return language, tts_generator, warning


def parse_bona_fide_metadata(path, root_dir):
    # Purpose: Extracts bona-fide metadata from the dataset folder structure when it is available.
    parts = path.relative_to(root_dir).parts
    language = parts[0] if len(parts) >= 2 else "unknown"
    warning = "" if language != "unknown" else "bona_fide language not reliably derivable from path"
    return language, "bona_fide", warning


def scan_audio_files(root_dir, label, class_name, source_kind):
    # Purpose: Scans a dataset folder and builds one metadata row for each supported audio file.
    root_dir = Path(root_dir)
    rows = []
    if not root_dir.exists():
        print("Directory missing:", root_dir)
        return pd.DataFrame(rows, columns=MANIFEST_COLUMNS)
    # Purpose: Iterates over this collection to build the next table, feature set, or experiment result
    # Purpose: consistently.
    for path in sorted(root_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
            continue
        if source_kind == "synthetic":
            language, tts_generator, warning = parse_synthetic_metadata(path, root_dir)
        else:
            language, tts_generator, warning = parse_bona_fide_metadata(path, root_dir)
        rows.append({
            "path": str(path.resolve()),
            "relative_path": str(path.relative_to(root_dir)),
            "label": int(label),
            "class_name": class_name,
            "language": language,
            "tts_generator": tts_generator,
            "file_size_mb": path.stat().st_size / (1024 * 1024),
            "metadata_warning": warning,
        })
    return pd.DataFrame(rows, columns=MANIFEST_COLUMNS)


## 4. Audit and Frozen Split


In [5]:
# Purpose: 4. Audit and frozen split shared with the MLP workflow.
def audio_info(path):
    try:
        info = sf.info(path)
        return {"path": str(path), "audio_exists": True, "native_sample_rate": int(info.samplerate), "channels": int(info.channels), "frames": int(info.frames), "duration_seconds": float(info.frames) / float(info.samplerate), "audio_info_error": ""}
    except Exception as exc:
        return {"path": str(path), "audio_exists": Path(path).exists(), "native_sample_rate": np.nan, "channels": np.nan, "frames": np.nan, "duration_seconds": np.nan, "audio_info_error": str(exc)}


def split_config_matches(path):
    if not path.exists():
        return False
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file).get("final_split_version") == FINAL_SPLIT_VERSION


split_config_path = MANIFESTS_DIR / "split_config.json"
train_manifest_path = MANIFESTS_DIR / "train_manifest.csv"
validation_manifest_path = MANIFESTS_DIR / "validation_manifest.csv"
test_manifest_path = MANIFESTS_DIR / "test_manifest.csv"
source_split_config = SOURCE_MLP_MANIFESTS_DIR / "split_config.json"
source_manifests = [SOURCE_MLP_MANIFESTS_DIR / name for name in ["train_manifest.csv", "validation_manifest.csv", "test_manifest.csv"]]

if split_config_matches(split_config_path) and all(path.exists() for path in [train_manifest_path, validation_manifest_path, test_manifest_path]):
    print("Loaded existing final CNN split:", MANIFESTS_DIR)
elif REUSE_MLP_SPLIT and split_config_matches(source_split_config) and all(path.exists() for path in source_manifests):
    # The exact MLP manifests are copied so cross-model comparisons use identical recordings.
    for source_path in [source_split_config, *source_manifests]:
        shutil.copy2(source_path, MANIFESTS_DIR / source_path.name)
    print("Reused the exact frozen MLP split for CNN:", SOURCE_MLP_MANIFESTS_DIR)
elif RUN_FULL_CNN_DATA_PREPARATION:
    synthetic_manifest = scan_audio_files(SYNTHETIC_AUDIO_DIR, 1, CLASS_NAMES[1], "synthetic")
    bona_fide_manifest = scan_audio_files(BONA_FIDE_AUDIO_DIR, 0, CLASS_NAMES[0], "bona_fide")
    if MAX_FILES_PER_CLASS is not None:
        synthetic_manifest = synthetic_manifest.sample(n=min(MAX_FILES_PER_CLASS, len(synthetic_manifest)), random_state=RANDOM_STATE)
        bona_fide_manifest = bona_fide_manifest.sample(n=min(MAX_FILES_PER_CLASS, len(bona_fide_manifest)), random_state=RANDOM_STATE)
    manifest = pd.concat([bona_fide_manifest, synthetic_manifest], ignore_index=True).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
    if manifest.empty:
        raise RuntimeError("No audio files were found.")
    audio_df = pd.DataFrame([audio_info(path) for path in manifest["path"]])
    manifest_audio = manifest.merge(audio_df, on="path", how="left", validate="one_to_one")
    manifest.to_csv(TABLES_DIR / "full_corrected_manifest.csv", index=False)
    manifest_audio.to_csv(TABLES_DIR / "manifest_with_audio_info.csv", index=False)
    manifest["class_name"].value_counts().rename_axis("class_name").reset_index(name="recordings").to_csv(TABLES_DIR / "class_counts.csv", index=False)
    manifest.groupby(["class_name", "language"]).size().reset_index(name="recordings").to_csv(TABLES_DIR / "class_language_counts.csv", index=False)
    manifest.loc[manifest["label"] == 1, "tts_generator"].value_counts().rename_axis("tts_generator").reset_index(name="synthetic_recordings").to_csv(TABLES_DIR / "synthetic_tts_generator_counts.csv", index=False)
    manifest_audio.groupby("class_name")["duration_seconds"].describe().reset_index().to_csv(TABLES_DIR / "duration_summary_by_class.csv", index=False)
    print("No reliable group key was derived. Using label-stratified splitting and documenting this limitation.")
    train_manifest, remaining = train_test_split(manifest, train_size=0.70, random_state=RANDOM_STATE, stratify=manifest["label"])
    validation_manifest, test_manifest = train_test_split(remaining, test_size=0.50, random_state=RANDOM_STATE, stratify=remaining["label"])
    train_manifest, validation_manifest, test_manifest = train_manifest.reset_index(drop=True), validation_manifest.reset_index(drop=True), test_manifest.reset_index(drop=True)
    for left_name, left, right_name, right in [("train", train_manifest, "validation", validation_manifest), ("train", train_manifest, "test", test_manifest), ("validation", validation_manifest, "test", test_manifest)]:
        if set(left["path"]) & set(right["path"]):
            raise RuntimeError(f"Path overlap between {left_name} and {right_name}.")
    train_manifest.to_csv(train_manifest_path, index=False)
    validation_manifest.to_csv(validation_manifest_path, index=False)
    test_manifest.to_csv(test_manifest_path, index=False)
    split_config = {"final_split_version": FINAL_SPLIT_VERSION, "random_state": RANDOM_STATE, "train_fraction": 0.70, "validation_fraction": 0.15, "test_fraction": 0.15, "group_key_used_or_null": None, "creation_method": "deterministic label-stratified fallback; exact MLP manifest reuse is preferred"}
    with open(split_config_path, "w", encoding="utf-8") as file:
        json.dump(split_config, file, indent=2)
else:
    print("Frozen manifests are missing. Run MLP notebook 00 first, or set RUN_FULL_CNN_DATA_PREPARATION = True intentionally.")


Loaded existing final CNN split: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/cnn/manifests


## 5. One MFCC Cache, Scaler and Class Weights


In [ ]:
# Purpose: 5. One 2-D MFCC cache, train-only normalisation and class weights.
def load_audio_fixed(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    if len(audio) > TARGET_SAMPLES:
        audio = audio[:TARGET_SAMPLES]
    elif len(audio) < TARGET_SAMPLES:
        audio = np.pad(audio, (0, TARGET_SAMPLES - len(audio)))
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio))
    return audio.astype(np.float32)


def extract_mfcc_map(path):
    # Shape is coefficient x time. The channel dimension is added when caches are loaded.
    audio = load_audio_fixed(path)
    return librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window="hann",
        center=True,
    ).astype(np.float32)


def build_feature_memmap(split_manifest, split_name):
    # A memory-mapped cache avoids holding every 2-D MFCC map in RAM at once.
    if split_manifest.empty:
        raise RuntimeError(f"The {split_name} manifest is empty.")
    first_map = extract_mfcc_map(split_manifest.iloc[0]["path"])
    feature_path = CACHE_DIR / f"X_{split_name}.npy"
    features = np.lib.format.open_memmap(
        feature_path,
        mode="w+",
        dtype=np.float32,
        shape=(len(split_manifest), *first_map.shape),
    )
    labels = np.empty(len(split_manifest), dtype=np.int64)
    skipped = []
    for row_index, row in split_manifest.reset_index(drop=True).iterrows():
        try:
            mfcc_map = first_map if row_index == 0 else extract_mfcc_map(row["path"])
            if mfcc_map.shape != first_map.shape:
                raise ValueError(f"Expected MFCC shape {first_map.shape}, got {mfcc_map.shape}")
            features[row_index] = mfcc_map
            labels[row_index] = int(row["label"])
        except Exception as exc:
            skipped.append({"split": split_name, "row_index": int(row_index), "path": row["path"], "error": str(exc)})
    features.flush()
    if skipped:
        pd.DataFrame(skipped).to_csv(TABLES_DIR / f"mfcc_extraction_skipped_{split_name}.csv", index=False)
        raise RuntimeError(f"MFCC extraction failed for {len(skipped)} {split_name} files.")
    np.save(CACHE_DIR / f"y_{split_name}.npy", labels)
    return features, labels


def normalise_memmap_in_place(features, coefficient_mean, coefficient_std, chunk_size=256):
    # Statistics have shape (1, n_mfcc, 1) and are learned from training data only.
    for start in range(0, len(features), chunk_size):
        stop = min(start + chunk_size, len(features))
        features[start:stop] = (features[start:stop] - coefficient_mean) / coefficient_std
    features.flush()


feature_config_path = CACHE_DIR / "feature_config.json"
cache_is_current = False
if feature_config_path.exists():
    with open(feature_config_path, "r", encoding="utf-8") as file:
        cache_is_current = json.load(file).get("final_split_version") == FINAL_SPLIT_VERSION

if cache_is_current:
    print("Existing final CNN cache is current:", CACHE_DIR)
elif RUN_FULL_CNN_DATA_PREPARATION and all(path.exists() for path in [train_manifest_path, validation_manifest_path, test_manifest_path]):
    train_manifest = pd.read_csv(train_manifest_path)
    validation_manifest = pd.read_csv(validation_manifest_path)
    test_manifest = pd.read_csv(test_manifest_path)
    X_train, y_train = build_feature_memmap(train_manifest, "train")
    X_validation, y_validation = build_feature_memmap(validation_manifest, "validation")
    X_test, y_test = build_feature_memmap(test_manifest, "test")

    # Per-coefficient statistics preserve the coefficient-by-time layout and prevent leakage.
    coefficient_mean = np.asarray(X_train.mean(axis=(0, 2), dtype=np.float64), dtype=np.float32)[None, :, None]
    coefficient_std = np.asarray(X_train.std(axis=(0, 2), dtype=np.float64), dtype=np.float32)[None, :, None]
    coefficient_std = np.where(coefficient_std < 1e-8, 1.0, coefficient_std).astype(np.float32)
    normalise_memmap_in_place(X_train, coefficient_mean, coefficient_std)
    normalise_memmap_in_place(X_validation, coefficient_mean, coefficient_std)
    normalise_memmap_in_place(X_test, coefficient_mean, coefficient_std)
    np.savez(CACHE_DIR / "mfcc_normalisation_stats.npz", mean=coefficient_mean, std=coefficient_std)

    labels = np.unique(y_train)
    values = compute_class_weight(class_weight="balanced", classes=labels, y=y_train)
    class_weights = {int(label): float(weight) for label, weight in zip(labels, values)}
    train_manifest.to_csv(CACHE_DIR / "train_metadata.csv", index=False)
    validation_manifest.to_csv(CACHE_DIR / "validation_metadata.csv", index=False)
    test_manifest.to_csv(CACHE_DIR / "test_metadata.csv", index=False)
    with open(CONFIGS_DIR / "class_weights.json", "w", encoding="utf-8") as file:
        json.dump({"class_weights": {str(key): value for key, value in class_weights.items()}, "source": "canonical y_train only", "final_split_version": FINAL_SPLIT_VERSION}, file, indent=2)
    feature_config = {
        "final_split_version": FINAL_SPLIT_VERSION,
        "representation": "2-D MFCC coefficient-by-time map; singleton channel added at load time",
        "mfcc_shape_without_channel": [int(value) for value in X_train.shape[1:]],
        "normalisation": "per-MFCC-coefficient mean/std fitted on X_train only",
        "center_frames": True,
        "test_features_prepared_but_not_evaluated": True,
    }
    with open(feature_config_path, "w", encoding="utf-8") as file:
        json.dump(feature_config, file, indent=2)
else:
    print("Final CNN cache is missing or outdated. Set RUN_FULL_CNN_DATA_PREPARATION = True to build it intentionally.")
